<div style="display: flex; align-items: center;">
    <h1>The AgMIP calibration protocol with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

The AgMIP calibration protocol provides a structured approach to calibrating crop models using multiple types of observations. It addresses three central challenges: combining different observation types, selecting which parameters to estimate, and finding parameter values that provide a good fit without unnecessary model complexity.

This notebook investigates the merits of gradient-based calibration, and how the protocol's calculation steps can be implemented with **diffWOFOST**, a differentiable implementation of WOFOST. In a differentiable model the simulated observations are a smooth function of the parameters, so the sum of squares has a gradient and the optimiser can follow it. We apply the protocol to the public French winter-wheat dataset from AgMIP Calibration Phase IV.

The protocol consists of eight steps. Steps 1–5 document the model-specific knowledge needed for calibration: default parameter values, observed and simulated variables, variable groups, major parameters, and candidate parameters. Steps 6–8 perform parameter estimation and selection, a joint weighted least-squares refit, and goodness-of-fit evaluation.

The reference implementation, `CroptimizR::run_protocol_agmip`, uses multi-start Nelder–Mead optimization. In this notebook we use gradient-based optimization while preserving the protocol's objective functions, parameter-selection criteria, and refitting procedure.

We first calibrate phenology, followed by biomass, plant nitrogen, yield, and grain protein. This ordering reflects the limited feedback from later processes onto phenological development, while the final weighted least-squares step allows all selected parameters to be estimated jointly.


## Dataset

AgMIP Calibration Phase IV uses the French winter-wheat cultivar Apache, evaluated in Arvalis variety trials. The dataset covers 22 site-years, divided into 14 calibration environments and eight evaluation environments. The two sets are separated by site and year, so the evaluation environments are independent of the calibration environments. The public data are available in [sbuis/AgMIP-Calibration-Phase-IV](https://github.com/sbuis/AgMIP-Calibration-Phase-IV):

The input file covers all 22 site-years, with weather joined from the relevant station sheets. The observation file contains only the 14 calibration environments. Observations from the eight evaluation environments are not included in the public repository. They are used by the AgMIP organizers to assess submitted simulations, as described by Wallach et al. (2024, 2025). Consequently, this notebook can evaluate calibration performance but cannot independently assess predictive performance on the official evaluation set.

One limitation concerns emergence. The units table defines an observation variable, `Date_BBCH10`, but the observation file does not contain this column. No emergence dates are available in the downloaded data, so emergence cannot be calibrated directly using these observations.

A separate synthetic dataset is available in [sbuis/AgMIP_calibration_PhaseIV_step2_synthetic_experiment](https://github.com/sbuis/AgMIP_calibration_PhaseIV_step2_synthetic_experiment). It represents the same environments simulated with STICS, for which the generating parameters are known. This dataset is useful for testing parameter recovery and calibration methodology under controlled conditions, but it is distinct from the real-data experiment used to prepare a WOFOST entry for the AgMIP evaluation.

## Applying the protocol to WOFOST

The AgMIP calibration protocol is model-independent: it specifies how to select observations and parameters, estimate parameters, select candidate parameters, and evaluate the calibrated model. Applying it to WOFOST nevertheless requires some adaptations because the available observations and model states do not match those of the original STICS example in every respect.

The main adaptations are:

- **Phenology:** BBCH55 is used as a proxy for anthesis (development stage 1), and BBCH90 represents maturity (development stage 2). BBCH30 is not fitted because it cannot be mapped directly to a distinct WOFOST development-stage threshold. The observation file also lacks emergence dates (BBCH10), so `TSUMEM` remains fixed at its Arminda default.
- **Observed variables:** Biomass is fitted on a logarithmic scale (`ln(TAGP)`). Yield is represented by `TWSO`, plant nitrogen by harvest nitrogen as a percentage of `TAGP`, and grain protein by 5.7 times the nitrogen percentage of `TWSO`. Grain number and ear number are not fitted because WOFOST does not provide the corresponding states.
- **Initial parameters and selection:** Calibration starts from the Arminda cultivar parameters. `TSUM1` and `TSUM2` are estimated to account for differences in phenology. Other parameters are classified as major parameters or candidates; candidates are retained according to the protocol's AICc-based selection procedure.
- **Optimization:** The reference implementation uses multi-start Nelder–Mead, whereas this notebook uses Adam. The protocol does not prescribe a particular optimizer, so the key requirement is to preserve the intended objective functions, candidate-selection procedure, and final weighted refit. The optimization settings and convergence diagnostics are described in the relevant sections.
- **Evaluation:** The 14 public site-years are used for calibration. The eight official evaluation environments can be simulated, but their observations are not publicly available, so independent evaluation against those observations is not possible here.

The hydraulic properties, rooting depth, initial mineral nitrogen, organic-matter properties, and fertilizer transformations are specified as fixed WOFOST inputs rather than calibrated parameters. Their assumptions are documented in the soil and nitrogen sections.

The notebook follows the structure of the AgMIP protocol while documenting the adaptations required by WOFOST and the available data. Results should be interpreted in light of these adaptations, particularly the use of heading as a proxy for anthesis and the absence of independent evaluation observations.

## Setup

The next cell installs `diffWOFOST` from the `agmip-calibration-protocol` branch, including its PCSE dependency, and loads the French AgMIP files. The winter-wheat parameter file is downloaded on first use if it is not already available locally.

### Optimization settings

We use Adam with the same numbers of initial parameter sets as the reference example: 20 Latin-hypercube starts for groups with two major parameters, five for groups with one major parameter, and five for each candidate. For candidate fits, one start uses the best parameter vector from the preceding fit; the remaining starts are generated by Latin-hypercube sampling.

The final weighted refit uses 20 starts, following the default in the reference `CroptimizR::run_protocol_agmip` implementation. The first starts from the parameter estimates obtained after sequential calibration and candidate selection, and the second from the default parameter values. The remaining starts explore the parameter bounds.

The initial learning rate is 0.25 for phenology and 0.12 for the later groups. If no new best objective is found for three steps, the optimizer returns to the best parameter vector, halves the learning rate, and resets Adam's momentum. This can happen up to four times. At the minimum learning rate, the search stops after ten steps without improvement, subject to a maximum of 80 steps. Candidate selection uses the best objective found for each fit.

These settings define the optimization strategy; they do not guarantee convergence. Results should therefore be interpreted alongside the convergence diagnostics and the completion status of each search.

### Saved results and reproducibility

By default, `DO_NOT_RERUN` is set to `True`, so the notebook loads the saved files `agmip_calibration_results.json` and `agmip_nelder_mead_results.json` from the notebook directory. These files contain the search results available on 26 October 2026.

The saved Nelder–Mead search had completed all groups. The Adam search had completed all groups through the protein candidate `NRESIDLV`, but did not complete the candidate `NRESIDST`: the run ended at step 37. The candidate `TCNT` was not started. Consequently, the Adam candidate-selection sequence is incomplete, and its results should not be treated as the final protocol calibration.

The saved Adam weighted-refit result is the best start completed before the run ended; it is not a completed multi-start weighted refit. No start of the saved Nelder–Mead weighted refit had finished.

Set `DO_NOT_RERUN = False` to run the searches again. A completed Adam search overwrites the Adam results file; the saved Nelder–Mead results remain unchanged. To reproduce the comparison fully, ensure that both methods have completed the same candidate-selection sequence and weighted refit.

In [ ]:
!pip install "diffwofost @ git+https://github.com/WUR-AI/diffWOFOST.git@agmip-calibration-protocol" matplotlib openpyxl

import datetime as dt
import json
import io
import tempfile
import urllib.request
import warnings
from pathlib import Path

import pandas as pd
import torch
from pcse.input import YAMLAgroManagementReader
from pcse.input import YAMLCropDataProvider
from pcse.models import Wofost81_NWLP_MLWB_SNOMIN

from diffwofost.physical_models.config import ComputeConfig
from diffwofost.physical_models.config import Configuration
from diffwofost.physical_models.crop.phenology import DVS_Phenology
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.parameter_providers import ParameterProvider

warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

# True loads the stored search and skips the Adam fits. False repeats them
# and overwrites agmip_calibration_results.json.
DO_NOT_RERUN = True
PATIENCE = 10
LR_PATIENCE = 3
LR_FACTOR = 0.5
LR_CUTS = 4
MIN_DELTA = 1.0e-4
PHENO_LR = 0.25


def rate_decision(score, best_ss, waited, cuts, rewound):
    """Step, halve the learning rate, or stop.

    Three steps without a new best return to the best parameters and halve
    the rate, up to four times. At the smallest rate the search stops after
    PATIENCE steps with no new best. The evaluation just after a rewind is
    the best point itself, so it does not count as a stall.
    """
    if rewound:
        return 0, cuts, False, "step"
    if score < best_ss - MIN_DELTA:
        return 0, cuts, False, "improve"
    waited += 1
    limit = PATIENCE if cuts >= LR_CUTS else LR_PATIENCE
    if waited < limit:
        return waited, cuts, False, "step"
    if cuts < LR_CUTS:
        return 0, cuts + 1, True, "decay"
    return waited, cuts, False, "stop"
RESULTS_NAME = "agmip_calibration_results.json"
RESULTS_URL = (
    "https://raw.githubusercontent.com/WUR-AI/diffWOFOST/"
    "agmip-calibration-protocol/docs/notebooks/agmip_calibration_results.json"
)
NELDER_NAME = "agmip_nelder_mead_results.json"
NELDER_URL = (
    "https://raw.githubusercontent.com/WUR-AI/diffWOFOST/"
    "agmip-calibration-protocol/docs/notebooks/agmip_nelder_mead_results.json"
)


def results_file():
    """Results beside this notebook, or in the working directory on Colab."""
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        notebook_dir = base / "docs" / "notebooks"
        if notebook_dir.is_dir():
            return notebook_dir / RESULTS_NAME
    return here / RESULTS_NAME


def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, (str, bool)) or value is None:
        return value
    if isinstance(value, int):
        return int(value)
    return float(value)


def load_stored(name, url):
    """JSON beside this notebook, downloaded once if it is missing."""
    path = results_file().with_name(name)
    if not path.exists():
        print(f"No results file at {path}. Downloading the stored search.")
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    print(f"Loaded {path.name}.")
    return json.loads(path.read_text())


def load_results():
    """Stored parameters, sums of squares and AICc rows."""
    return load_stored(RESULTS_NAME, RESULTS_URL)


def load_nelder():
    """Stored Nelder-Mead comparison. Independent of DO_NOT_RERUN."""
    return load_stored(NELDER_NAME, NELDER_URL)


def save_results(payload):
    path = results_file()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(_jsonable(payload), indent=2) + "\n")
    print(f"Stored calibration results in {path}.")


DATA_URL = "https://raw.githubusercontent.com/sbuis/AgMIP-Calibration-Phase-IV/main/data"


def crop_directory():
    """Winter-wheat parameter file shipped with the SNOMIN notebook."""
    here = Path.cwd().resolve()
    relative = (
        Path("docs/notebooks/snomin_calibration_data/crop"),
        Path("snomin_calibration_data/crop"),
        Path("crop"),
    )
    for base in (here, *here.parents):
        for name in relative:
            candidate = base / name
            if (candidate / "winterwheat.yaml").exists():
                return candidate
    destination = here / "snomin_calibration_data" / "crop"
    destination.mkdir(parents=True, exist_ok=True)
    names = ("winterwheat.yaml", "crops.yaml")
    for ref in ("agmip-calibration-protocol", "main"):
        base = (
            "https://raw.githubusercontent.com/WUR-AI/diffWOFOST/"
            f"{ref}/docs/notebooks/snomin_calibration_data/crop"
        )
        try:
            for name in names:
                urllib.request.urlretrieve(f"{base}/{name}", destination / name)
        except Exception:
            continue
        if (destination / "winterwheat.yaml").exists():
            return destination
    raise FileNotFoundError("winterwheat.yaml is not next to this notebook")


def download(name):
    with urllib.request.urlopen(f"{DATA_URL}/{name}") as response:
        return response.read()


observations = pd.read_csv(io.BytesIO(download("cal_4_obs_French_A.txt")), sep=r"\s+")
management = pd.read_csv(
    io.BytesIO(download("cal_4_input_data_French_A.txt")), sep=r"\s+", encoding="utf-8-sig"
)
units = pd.read_csv(io.BytesIO(download("cal_4_obs_French_units.csv")), sep=";", encoding="cp1252")
weather_book = pd.ExcelFile(io.BytesIO(download("cal_4_weather.xlsx")))
print(f"Calibration rows: {len(observations)}. Site-years in the input file: {len(management)}.")
print(f"Weather stations: {', '.join(weather_book.sheet_names)}.")
units


## Steps 1–5: applying the AgMIP protocol to WOFOST

**Step 1 — Default parameters.** We use the Arminda cultivar parameters from the WOFOST 8.1 winter-wheat parameter file as the starting point. Apache, the target cultivar, is a French soft winter wheat. The protocol recommends starting from a cultivar of a similar maturity class and calibrating parameters to account for differences between cultivars. The WOFOST 8.1 nitrogen parameters were estimated for Arminda (Berghuijs et al., 2024). Arminda uses vernalization (`IDSL = 2`).

**Steps 2–3 — Observations and variable groups.** Only observations with a corresponding simulated variable in WOFOST can be used directly for calibration. The available observations are assigned to variable groups according to the model outputs they constrain. Observations without a suitable simulated equivalent, or without the required measurements, are excluded.

**Step 4 — Major parameters.** Each variable group is calibrated first using a small set of major parameters with clear mechanistic links to the observations. For phenology, `TSUM1` and `TSUM2` control development towards anthesis and maturity, respectively. Because the observations record heading (BBCH55) rather than anthesis, heading is used as a proxy for anthesis.

**Step 5 — Candidate parameters.** Additional candidate parameters are tested sequentially and retained according to the protocol's AICc-based selection criterion. For phenology, vernalization is tested before daylength because the crops are autumn-sown and the calibration environments experience different winter temperatures. Daylength is tested afterwards; the sites' relatively narrow latitude range (approximately 47–50°N) limits, but does not eliminate, variation in photoperiod.

The major parameters selected for each variable group, and the observations used to constrain them, are summarized below.

| Variable group | Observations used | Major parameters | Rationale |
|---|---|---|---|
| Phenology | Heading (BBCH55) and maturity (BBCH90) dates | `TSUM1`, `TSUM2` | Control development towards anthesis and maturity, respectively. Heading is used as a proxy for anthesis. |
| Biomass | Log-transformed above-ground biomass, `ln(TAGP)` | `AMAX_REF`, `SPAN` | Control reference assimilation capacity and leaf lifespan, influencing biomass accumulation. |
| Plant nitrogen | Harvest nitrogen as a percentage of above-ground biomass | `RNUPTAKEMAX` | Sets the maximum rate of nitrogen uptake. |
| Yield | Total weight of storage organs (`TWSO`) | `REALLOC_STEM_FRACTION` | Controls the fraction of stem biomass available for reallocation to grain. |
| Grain protein | Grain protein estimated as 5.7 times the nitrogen percentage of `TWSO` | `NMAXSO` | Sets the maximum nitrogen concentration in grain. |

For grain protein, three additional candidate parameters are considered: `NRESIDLV`, `NRESIDST`, and `TCNT`. These govern aspects of nitrogen retention in vegetative tissues and nitrogen transfer to grain. They are tested sequentially to assess whether they improve the fit sufficiently to justify the additional parameter complexity, as evaluated using AICc.

Some observations cannot be used directly because the required measurements are unavailable or WOFOST does not provide corresponding simulated quantities. For example, emergence dates (BBCH10) are absent from the observation file, while grain number and ear number are not fitted because WOFOST does not provide corresponding simulated states.

In [ ]:
variables = pd.DataFrame(
    [
        ["Date_BBCH30", "none", "phenology", "", "Stem elongation is not a WOFOST stage."],
        ["Date_BBCH55", "day when DVS reaches 1", "phenology", 1, "Heading, used as anthesis."],
        ["Date_BBCH90", "day when DVS reaches 2", "phenology", 1, "Maturity."],
        ["Biomass", "TAGP", "plant_biomass", 2, "g m-2 in the file, kg ha-1 in WOFOST."],
        ["N_in_biomassHarvest", "crop N / TAGP", "plant_N_content", 3, "Percent. Needs SNOMIN."],
        ["Grain_Yield", "TWSO", "yield", 4, "g m-2 in the file, kg ha-1 in WOFOST."],
        ["ProteinContentGrain", "storage-organ N / TWSO", "seed_protein", 5, "Percent."],
        ["EarsPerSqM", "none", "", "", "WOFOST does not simulate ear number."],
        ["Grain_Number", "none", "", "", "WOFOST does not simulate grain number."],
        ["LAI", "none", "", "", "Not in the observation file."],
    ],
    columns=["Observed", "Simulated", "Group", "Order", "Note"],
)
parameters = pd.DataFrame(
    [
        ["phenology", "major", "TSUM1", "Temperature sum, emergence to anthesis. Moves BBCH55."],
        ["phenology", "major", "TSUM2", "Temperature sum, anthesis to maturity. Moves BBCH90."],
        ["phenology", "candidate", "VERNSAT", "Saturated vernalization requirement."],
        ["phenology", "candidate", "VERNBASE", "Base vernalization requirement."],
        ["phenology", "candidate", "DLO", "Optimum daylength."],
        ["plant_biomass", "major", "AMAX_REF", "Assimilation level. One of the two allowed majors."],
        ["plant_biomass", "major", "SPAN", "Leaf lifespan. Sets how long biomass keeps accumulating."],
        ["plant_biomass", "candidate", "TDWI", "Initial biomass. Early-season residual."],
        ["plant_biomass", "candidate", "RGRLAI", "Early canopy closure. Tried on biomass only; there is no LAI to fit."],
        ["plant_N_content", "major", "RNUPTAKEMAX", "Maximum daily nitrogen uptake."],
        ["plant_N_content", "candidate", "NMAXSO", "Maximum nitrogen concentration in the grain."],
        ["yield", "major", "REALLOC_STEM_FRACTION", "Stem biomass that can move into the grain."],
        ["yield", "candidate", "CVO", "Conversion efficiency into storage organs."],
        ["seed_protein", "major", "NMAXSO", "Same parameter as above, if it was not already selected."],
        ["seed_protein", "candidate", "NRESIDLV", "Residual nitrogen in leaves. Lowering it releases more nitrogen to the grain."],
        ["seed_protein", "candidate", "NRESIDST", "Residual nitrogen in stems."],
        ["seed_protein", "candidate", "TCNT", "Days over which translocatable nitrogen moves into the grain."],
    ],
    columns=["Group", "Role", "Parameter", "Why this one"],
)
display(variables)
display(parameters)


## Modeling choices and assumptions

The AgMIP protocol defines how to calibrate the crop model, but applying it also requires decisions about the error metric, fertilizer inputs, soil properties, and parameters that can be identified from the available observations. The choices below describe how these issues are handled in this WOFOST implementation.

### 1. Sum of squares: why biomass is fitted on a logarithmic scale

The calibration minimizes a sum of squared errors within each observation group. AICc is then used to compare candidate models, balancing goodness of fit against the number of fitted parameters. The error scale matters because it determines which observations have the greatest influence on the fit.

Consider two biomass observations: 100 and 2,000 g m⁻². If the model overestimates each by 20%, the absolute errors are 20 and 400 g m⁻², respectively. After squaring, these contribute 400 and 160,000 to the sum of squares. The larger observation therefore dominates the objective, even though the relative errors are identical.

To reduce this effect, biomass is fitted on a logarithmic scale. The residual is the difference between the logarithms of observed and simulated biomass: `log(TAGP_obs) - log(TAGP_sim)`.
This measures the difference in relative terms: for small errors, the log residual is approximately the relative error. Consequently, a similar percentage error receives a similar weight across the season, rather than larger observations automatically dominating because of their magnitude.

The other observation groups remain on their original scales:

- **Phenology:** dates are fitted in days. A three-day error has the same interpretation regardless of when the stage occurs.
- **Yield:** grain yield is fitted in g m⁻².
- **Plant nitrogen:** harvest nitrogen content is fitted as a percentage of above-ground biomass.
- **Grain protein:** protein content is fitted as a percentage.

Yield, plant nitrogen, and grain protein provide one harvest observation per environment. Unlike the repeated biomass measurements, they do not have a seasonal increase in magnitude that motivates a logarithmic transformation.

### 2. Fertilizer: translating trial records into SNOMIN inputs

The trial input file records fertilizer applications by date, product, and nitrogen rate (kg N ha⁻¹). SNOMIN represents an application using an amount, application depth, organic-matter fraction, C:N ratio, initial age, and the fractions of nitrogen supplied as ammonium and nitrate.

The fertilizer products considered here are treated as mineral fertilizers, so the organic-matter fraction is set to zero. The C:N ratio and initial age therefore have no effect on these applications. The ammonium and nitrate fractions must sum to one so that the recorded nitrogen rate is fully allocated between the two mineral nitrogen pools.

The product-specific assumptions are:

- **Ammonitrate:** treated as ammonium nitrate, with half the nitrogen assigned to ammonium and half to nitrate (`f_NH4N = 0.5`, `f_NO3N = 0.5`).
- **18_46:** treated as diammonium phosphate, with its nitrogen assigned to ammonium (`f_NH4N = 1.0`, `f_NO3N = 0.0`).
- **Solution azotée:** treated as a urea–ammonium nitrate solution, with approximately half its nitrogen from urea, one-quarter from ammonium, and one-quarter from nitrate. Because SNOMIN does not explicitly represent urea hydrolysis, the urea share is assigned to ammonium (`f_NH4N = 0.75`, `f_NO3N = 0.25`). This is a simplification: it removes the short delay associated with urea hydrolysis and does not explicitly represent ammonia volatilization.

The input file does not specify fertilizer incorporation depth. The implementation assumes 10 cm. Because SNOMIN applies the amendment within the soil-layer structure, the effective placement depends on the thickness of the uppermost layer, which is 25–30 cm in these profiles.

These choices translate the trial records into the model's input format; they are assumptions about fertilizer behavior, not additional calibrated parameters.

### 3. Soil: deriving hydraulic properties and fixing nitrogen assumptions

The soil input file describes each horizon using its thickness, texture, bulk density, field capacity, wilting point, stone content, pH, and organic matter. WOFOST requires soil hydraulic properties to simulate water availability and crop growth, while SNOMIN additionally requires assumptions about soil organic matter and nitrogen cycling.

**Hydraulic properties.** The implementation uses the Wösten pedotransfer function to estimate the parameters of a van Genuchten-type water-retention curve and the associated hydraulic conductivity. These estimates are based on soil texture, organic matter, bulk density, and whether the horizon is the topsoil. The measured field capacity and wilting point are compared with the corresponding values derived from the pedotransfer function as a diagnostic check; they do not directly determine the fitted retention curve.

**Soil organic matter and nitrogen.** SNOMIN requires the C:N ratio of the initial soil organic matter (`CNRatioSOMI`) to represent the nitrogen content of that pool and its contribution to mineralization. The input file supplies organic matter content but not the soil C:N ratio. The implementation therefore uses a fixed arable-soil value of approximately 10 across environments.

The apparent age of the initial organic matter is also unavailable. The implementation fixes it at 24 years, following the value used in the SoilGrids example.

Both quantities are held fixed rather than estimated separately for each environment. Otherwise, they could absorb errors caused by other model components or imperfect input data, making it difficult to distinguish soil assumptions from crop-parameter effects.

Soil hydraulic properties, rooting depth, initial mineral nitrogen, and organic-matter properties are therefore prescribed inputs in this calibration. Their uncertainty may affect the fitted crop parameters, but it is not estimated by the calibration procedure described here.

### 4. Why phenology is calibrated without the soil model

Phenology is calibrated first, following the protocol's sequence. WOFOST phenology is represented by the `DVS_Phenology` component, which calculates development stages from temperature and the relevant cultivar parameters without requiring the soil water and nitrogen modules.

The phenology group therefore uses `DVS_Phenology` alone. Once calibrated, its parameters are held fixed during calibration of biomass, plant nitrogen, yield, and grain protein. Those subsequent groups use the full crop model (`Wofost81`) together with the soil and nitrogen model (`SoilModuleWrapper_NWLP_MLWB_SNOMIN`).

This separation follows the intended calibration sequence: first establish the timing of crop development, then estimate parameters governing growth, nitrogen uptake, yield formation, and grain protein.

### 5. Why `TSUMEM` remains fixed

`TSUMEM` is the temperature sum required between sowing and emergence. `TSUM1` controls development from emergence to anthesis, and `TSUM2` controls development from anthesis to maturity.

The observation file does not contain emergence dates (BBCH10). The available phenology observations are heading (BBCH55), used as a proxy for anthesis, and maturity (BBCH90).

Without emergence observations, `TSUMEM` and `TSUM1` cannot be independently identified reliably from heading and maturity dates alone. Increasing `TSUMEM` while decreasing `TSUM1` by a similar amount can leave the total thermal-time requirement up to anthesis almost unchanged. The two parameters can therefore compensate for one another during calibration.

To avoid estimating poorly identifiable parameters, `TSUMEM` is held at its Arminda default of 120 °C day, while `TSUM1` and `TSUM2` are estimated from the available phenology observations.

## Step 6: calibrating and selecting phenology parameters

### 1. Fitting the major parameters

Phenology is the first variable group calibrated. We use heading (BBCH55) and maturity (BBCH90) dates from the 14 calibration site-years, giving 28 observations in total.

Arminda provides the reference parameterization for WOFOST. For phenology, however, `TSUM1` and `TSUM2` are estimated using 20 Latin-hypercube initializations, which explore different starting combinations within the permitted parameter bounds. Each initialization runs a separate optimization, and the best fit found is used for subsequent parameter selection. Parameters not selected for estimation remain at their Arminda defaults.

The two major parameters are:

- `TSUM1`: the temperature sum from emergence to anthesis, controlling the timing of heading.
- `TSUM2`: the temperature sum from anthesis to maturity, controlling the timing of maturity.

Because the observations record heading rather than anthesis, heading (BBCH55) is used as a proxy for anthesis.

### 2. Selecting additional parameters using AICc

A good fit should explain the observations without introducing unnecessary parameters. The Akaike Information Criterion (AIC) balances these two objectives. Assuming independent Gaussian errors with a common variance, it can be written, up to a constant shared by the candidate models, as

$$
\mathrm{AIC} = n\ln(\mathrm{SS}/n)+2p,
$$

where `n` is the number of observations, `SS` is the minimized sum of squares, and `p` is the number of estimated parameters. A lower AIC indicates a better balance between goodness of fit and model complexity.

Adding a parameter can reduce the sum of squares, but it also increases the penalty. The improvement in fit must be large enough to justify estimating the extra parameter.

Because the sample is relatively small, the protocol uses the corrected criterion, AICc:

$$
\mathrm{AICc} =
n\ln(\mathrm{SS}/n)+2p+
\frac{2p(p+1)}{n-p-1}.
$$

The final term increases the penalty when the number of observations is small relative to the number of parameters.

Here, `n = 28`. The correction is approximately 0.48 for the two major parameters and 1.0 for a model with three parameters. A candidate parameter is retained only if adding it reduces AICc.

### 3. Phenology calibration results

Both the Adam and Nelder–Mead searches retain only `TSUM1` and `TSUM2`. Adding `VERNSAT`, `VERNBASE`, or `DLO` does not reduce the sum of squares in the stored results, so the additional parameter complexity increases AICc.

| Search | Parameters retained | Sum of squares | AICc |
|---|---|---:|---:|
| Adam | `TSUM1`, `TSUM2` | 839.1 | 99.68 |
| Nelder–Mead | `TSUM1`, `TSUM2` | 837.9 | 99.64 |

The estimated values are similar between the two methods:

| Parameter | Arminda default | Adam | Nelder–Mead |
|---|---:|---:|---:|
| `TSUM1` (°C day) | 886 | 618.4 | 618.4 |
| `TSUM2` (°C day) | 860 | 956.1 | 961.4 |

The candidate parameters remain at their Arminda defaults: `VERNSAT` = 44 days, `VERNBASE` = 9 days, and `DLO` = 16.3 hours. Because freeing these parameters did not improve the fit in the stored searches, they were not retained.

These results illustrate the purpose of candidate selection: a parameter is not retained simply because it could influence phenology. It must improve the fit enough to justify the additional complexity.

### 4. Estimating the variance for weighted calibration

After parameter selection, the protocol estimates the residual error variance for each observation group. For phenology, this is calculated as

$$
\hat{\sigma}^2 = \frac{\mathrm{SS}}{n-p},
$$

where `p` is the number of parameters retained. This differs from the `SS/n` term used inside the AICc likelihood expression.

The estimated variance is used to scale each group's sum of squares in the final weighted least-squares objective. Each group contributes its sum of squares divided by its estimated error variance, so groups with larger residual variance receive less weight.

For phenology alone, dividing the sum of squares by a fixed variance does not change the parameter values that minimize it. The weighted objective therefore has the same optimum for this group on its own. Parameters can, however, change in the final joint refit when all selected groups contribute to the objective.

### 5. Smooth dates for optimization, actual dates for evaluation

Observed phenology dates correspond to the first day on which the crop reaches the relevant development stage. A hard threshold crossing is not necessarily differentiable with respect to model parameters, which makes it unsuitable for straightforward gradient-based optimization.

The calibration therefore uses a smooth approximation to the stage-crossing date, allowing gradients to guide the optimizer. Reported phenology errors, however, are calculated from the actual simulated crossing dates. This distinction enables gradient-based fitting while keeping the evaluation metric in calendar days.

In [ ]:
SHARPNESS = 40.0
BOUNDS = {
    "TSUM1": (400.0, 1400.0),
    "TSUM2": (400.0, 1400.0),
    "VERNSAT": (30.0, 90.0),
    "VERNBASE": (0.0, 20.0),
    "DLO": (12.0, 20.0),
}
MAJOR = ("TSUM1", "TSUM2")
CANDIDATES = ("VERNSAT", "VERNBASE", "DLO")
SCALARS = (
    "TSUMEM", "TBASEM", "TEFFMX", "TSUM1", "TSUM2", "IDSL", "DLO", "DLC",
    "DVSI", "DVSEND", "VERNSAT", "VERNBASE", "VERNDVS",
)


def parse_date(value):
    if pd.isna(value) or value == "NA":
        return None
    return dt.datetime.strptime(str(value), "%d/%m/%Y").date()


def station_weather(station):
    """Daily WOFOST weather. Radiation in the workbook is MJ m-2 day-1."""
    sheet = pd.read_excel(weather_book, f"station_{station}", header=None)
    latitude = float(str(sheet.iloc[0, -1]).split("Latitude =")[1].split("§")[0])
    altitude = float(str(sheet.iloc[0, -1]).split("Altitude =")[1])
    daily = sheet.iloc[5:].copy()
    daily.columns = [
        "DATECODE", "YYYY", "MM", "DD", "SRAD", "TMAX", "TMIN", "RAIN",
        "ET0", "WIND", "DEWP", "VPRS", "RHUM", "DAY",
    ]
    daily["DAY"] = pd.to_datetime(daily["DAY"]).dt.date
    return daily, latitude, altitude


def driving_rows(daily, latitude, altitude, start, end):
    window = daily[(daily["DAY"] >= start) & (daily["DAY"] <= end)]
    rows = []
    for record in window.itertuples(index=False):
        tmin = float(record.TMIN)
        tmax = float(record.TMAX)
        rows.append({
            "DAY": record.DAY,
            "LAT": latitude,
            "LON": 2.0,
            "ELEV": altitude,
            "IRRAD": float(record.SRAD) * 1.0e6,
            "TMIN": tmin,
            "TMAX": tmax,
            "TEMP": 0.5 * (tmin + tmax),
            "RAIN": float(record.RAIN) / 10.0,
            "VAP": 10.0,
            "WIND": 2.0,
        })
    return rows


def agro_calendar(start, end):
    text = f"""
AgroManagement:
- {start.isoformat()}:
    CropCalendar:
        crop_name: winterwheat
        variety_name: Arminda
        crop_start_date: {start.isoformat()}
        crop_start_type: sowing
        crop_end_date: {end.isoformat()}
        crop_end_type: harvest
        max_duration: 360
    TimedEvents: null
    StateEvents: null
"""
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as handle:
        handle.write(text)
        path = Path(handle.name)
    try:
        return YAMLAgroManagementReader(str(path))
    finally:
        path.unlink(missing_ok=True)


phenology_rows = (
    observations.dropna(subset=["Date_BBCH55", "Date_BBCH90"])
    .groupby("Number", as_index=False)
    .tail(1)
)
environments = []
for record in phenology_rows.itertuples(index=False):
    site = management.loc[management["Number"] == record.Number].iloc[0]
    sowing = parse_date(record.SowingDate)
    heading = parse_date(record.Date_BBCH55)
    maturity = parse_date(record.Date_BBCH90)
    harvest = parse_date(record.HarvestDate) or maturity
    daily, latitude, altitude = station_weather(int(site.Station))
    end = min(harvest + dt.timedelta(days=45), daily["DAY"].max())
    rows = driving_rows(daily, latitude, altitude, sowing, end)
    if not rows or rows[-1]["DAY"] < maturity:
        print(f"Skipping environment {record.Number}: weather does not cover maturity.")
        continue
    environments.append({
        "number": int(record.Number),
        "site": record.Site,
        "year": int(record.HarvestYear),
        "rows": rows,
        "agro": agro_calendar(sowing, rows[-1]["DAY"]),
        "heading": (heading - sowing).days,
        "maturity": (maturity - sowing).days,
    })

held_out = sorted(set(management["Number"]) - set(phenology_rows["Number"]))
print(f"Calibration environments with heading and maturity: {len(environments)}.")
print(f"Held-out environment numbers, observations not public: {held_out}.")

crop = YAMLCropDataProvider(
    Wofost81_NWLP_MLWB_SNOMIN, fpath=str(crop_directory()), force_reload=True
)
provider = ParameterProvider(cropdata=crop)
provider.set_active_crop("winterwheat", "Arminda", "sowing", "harvest")
for name in list(SCALARS) + ["DTSMTB", "VERNRTB"]:
    value = provider[name]
    if not torch.is_tensor(value):
        value = torch.tensor(value, dtype=torch.float64)
    provider.set_override(name, value, check=False)
defaults = {name: float(provider[name]) for name in list(BOUNDS) + ["TSUMEM", "IDSL"]}
print(
    "Arminda defaults: "
    + ", ".join(f"{name} {defaults[name]:.1f}" for name in ("TSUM1", "TSUM2", "VERNSAT", "VERNBASE", "DLO"))
)

config = Configuration(CROP=DVS_Phenology, OUTPUT_VARS=["DVS"])


In [ ]:
class BoundedParameter(torch.nn.Module):
    """A parameter kept inside its protocol bounds."""

    def __init__(self, low, high, init_value):
        super().__init__()
        self.low = low
        self.high = high
        fraction = (init_value - low) / (high - low)
        fraction = min(max(fraction, 1.0e-4), 1.0 - 1.0e-4)
        self.raw = torch.nn.Parameter(torch.logit(torch.tensor(fraction, dtype=torch.float64)))

    def forward(self):
        return self.low + (self.high - self.low) * torch.sigmoid(self.raw)


class ParameterSet(torch.nn.Module):
    def __init__(self, names, start):
        super().__init__()
        self.boxes = torch.nn.ModuleDict({
            name: BoundedParameter(*BOUNDS[name], start[name]) for name in names
        })

    def forward(self):
        return {name: box() for name, box in self.boxes.items()}


def apply(active):
    for name in SCALARS:
        if name in active:
            provider.set_override(name, active[name], check=False)
        elif name in accepted:
            provider.set_override(name, torch.tensor(accepted[name], dtype=torch.float64), check=False)


def development(environment, active):
    apply(active)
    engine = Engine(config)
    engine.setup(provider, iter(environment["rows"]), environment["agro"])
    engine.run_till(environment["rows"][-1]["DAY"])
    return torch.stack([day["DVS"].reshape(()) for day in engine.get_output()])


def stage_error(dvs, threshold, observed_day):
    """Error in days. A stage past the season end uses the remaining development."""
    past = dvs > threshold + 0.05
    capped = torch.where(past, dvs.detach(), dvs)
    previous = torch.cat((capped.new_zeros(1), capped[:-1]))
    entered = torch.sigmoid(SHARPNESS * (capped - threshold))
    entered = entered - torch.sigmoid(SHARPNESS * (previous - threshold))
    entered = entered.clamp(min=0.0)
    days = torch.arange(capped.shape[0], dtype=capped.dtype)
    weight = entered.sum()
    day = (entered * days).sum() / weight.clamp(min=1.0e-6)
    calendar = day - observed_day
    unfinished = (threshold - dvs[-1]).clamp(min=0.0) / 0.02
    beyond = unfinished + (dvs.shape[0] - 1 - observed_day)
    return torch.where(weight > 0.2, calendar, beyond)


def sum_of_squares(active):
    total = torch.zeros((), dtype=torch.float64)
    for environment in environments:
        dvs = development(environment, active)
        heading = stage_error(dvs, 1.0, environment["heading"])
        maturity = stage_error(dvs, 2.0, environment["maturity"])
        total = total + heading.square() + maturity.square()
    return total


def aicc(ss, n_obs, n_param):
    return n_obs * torch.log(ss / n_obs) + 2 * n_param + (2 * n_param * (n_param + 1)) / (n_obs - n_param - 1)


def estimate(names, start, max_steps=80):
    """Adam, halving the rate when the sum of squares stalls."""
    pack = ParameterSet(names, start)
    lr = PHENO_LR
    optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
    best_ss = float("inf")
    best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
    waited, cuts, rewound = 0, 0, False
    for step in range(max_steps):
        optimizer.zero_grad()
        last = sum_of_squares(pack())
        score = float(last.detach())
        waited, cuts, rewound, action = rate_decision(score, best_ss, waited, cuts, rewound)
        if action == "improve":
            best_ss = score
            best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
        elif action == "decay":
            pack.load_state_dict(best_state)
            lr *= LR_FACTOR
            optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
            print(f"Learning rate {lr:.4g}. Back at the best step.")
            continue
        elif action == "stop":
            print(f"Early stopping at step {step}.")
            break
        last.backward()
        optimizer.step()
    else:
        print(f"Reached the ceiling of {max_steps} steps.")
    pack.load_state_dict(best_state)
    with torch.no_grad():
        values = {name: float(value) for name, value in pack().items()}
        ss = float(sum_of_squares(pack()))
    return values, ss


def hard_day(dvs, threshold):
    crossed = (dvs >= threshold).nonzero()
    if len(crossed) == 0:
        return None
    return int(crossed[0])


def stage_errors(values):
    """Signed errors in days, heading then maturity, one pair per environment."""
    heading, maturity = [], []
    with torch.no_grad():
        active = {name: torch.tensor(values[name], dtype=torch.float64) for name in values}
        for environment in environments:
            dvs = development(environment, active)
            for threshold, observed, store in (
                (1.0, environment["heading"], heading),
                (2.0, environment["maturity"], maturity),
            ):
                day = hard_day(dvs, threshold)
                store.append(None if day is None else day - observed)
    return heading, maturity


n_observations = 2 * len(environments)
if DO_NOT_RERUN:
    stored = load_results()
    history = stored["phenology_history"]
    accepted = {name: float(value) for name, value in stored["phenology_accepted"].items()}
    selected = list(stored["phenology_selected"])
    error_variance = float(stored["phenology_error_variance"])
    print("DO_NOT_RERUN is set, so the phenology search was loaded.")
    for row in history:
        print(f"{row['Parameters']}: SS {row['SS']:.1f}. AICc {row['AICc']:.2f}. {row['Kept']}.")
else:
    n_observations = 2 * len(environments)
    accepted = {name: defaults[name] for name in BOUNDS}
    history = []

    major_values, major_ss = estimate(MAJOR, accepted)
    accepted.update(major_values)
    selected = list(MAJOR)
    best_aicc = float(aicc(torch.tensor(major_ss), n_observations, len(selected)))
    history.append({"Parameters": ", ".join(selected), "SS": major_ss, "AICc": best_aicc, "Kept": "major"})
    print(f"Major parameters. SS {major_ss:.1f}. AICc {best_aicc:.2f}.")

    for candidate in CANDIDATES:
        trial = selected + [candidate]
        values, ss = estimate(trial, accepted)
        score = float(aicc(torch.tensor(ss), n_observations, len(trial)))
        keep = score < best_aicc
        history.append({"Parameters": ", ".join(trial), "SS": ss, "AICc": score, "Kept": "yes" if keep else "no"})
        print(f"{candidate}: AICc {score:.2f} against {best_aicc:.2f}. {'Kept' if keep else 'Left at the default'}.")
        if keep:
            accepted.update(values)
            selected = trial
            best_aicc = score

    final_ss = next(row["SS"] for row in reversed(history) if row["Kept"] != "no")
    error_variance = final_ss / (n_observations - len(selected))
    print(
        f"Step 7 weight for phenology, SS/(n-p) after step 6: {error_variance:.2f} day². "
        "One group makes weighted least squares the same fit."
    )
pd.DataFrame(history)


In [ ]:
def summarize(errors):
    series = pd.Series([error for error in errors if error is not None], dtype=float)
    if series.empty:
        return {"n": 0, "bias": None, "RMSE": None}
    return {
        "n": int(series.size),
        "bias": series.mean(),
        "RMSE": (series.pow(2).mean()) ** 0.5,
    }


default_heading, default_maturity = stage_errors({name: defaults[name] for name in BOUNDS})
final_heading, final_maturity = stage_errors(accepted)
metrics = pd.DataFrame(
    [
        {"Stage": "BBCH55", "Parameters": "Arminda default", **summarize(default_heading)},
        {"Stage": "BBCH55", "Parameters": "after step 6", **summarize(final_heading)},
        {"Stage": "BBCH90", "Parameters": "Arminda default", **summarize(default_maturity)},
        {"Stage": "BBCH90", "Parameters": "after step 6", **summarize(final_maturity)},
    ]
)
estimated = pd.DataFrame(
    {
        "Parameter": list(BOUNDS),
        "Role": ["selected" if name in selected else "default" for name in BOUNDS],
        "Default": [defaults[name] for name in BOUNDS],
        "Estimated": [accepted[name] for name in BOUNDS],
    }
)
display(metrics.round(2))
display(estimated.round(2))


## Step 6: calibrating biomass, nitrogen, yield and grain protein

After calibrating phenology, we hold the selected phenology parameters fixed while calibrating the remaining four variable groups individually. Each group has its own observations, major parameters, and candidate parameters. Candidate selection uses AICc to balance goodness of fit against model complexity.

### Observation scales

The observations must be expressed on the same scale as the corresponding WOFOST outputs.

- **Biomass:** The observed above-ground biomass (`TAGP`) is recorded in g m⁻², whereas WOFOST reports it in kg ha⁻¹. We convert the observations by multiplying by 10, then apply a logarithmic transformation. This makes the fitting objective more sensitive to relative errors than to absolute differences in biomass.
- **Yield:** Observed grain yield is recorded in g m⁻². WOFOST's storage-organ weight (`TWSO`) is converted from kg ha⁻¹ to g m⁻² by dividing by 10.
- **Plant nitrogen:** The target is the percentage of above-ground biomass nitrogen contained in the plant at harvest, including leaves, stems and grain.
- **Grain protein:** Grain protein is estimated as 5.7 times the nitrogen concentration of the storage organs (`TWSO`). Zero values in the observation file are treated as missing rather than as measured zero protein.

The soil hydraulic properties, rooting depth, and soil-nitrogen inputs remain fixed at the values specified in the soil section. They are not estimated during these crop-parameter calibration steps.

### Optimization and candidate selection

The parameters in each group are estimated using Adam, with an initial learning rate of 0.12. The learning rate is reduced when progress stalls: after three steps without a new best objective value, the search returns to the best solution found and halves the learning rate. This can happen up to four times. Optimization stops after ten steps without improvement at the smallest learning rate, or after 80 steps, whichever comes first.

The best solution found is used to evaluate the fit and calculate AICc. Additional candidate parameters are retained only when they reduce AICc sufficiently to justify the added complexity.

For each group, the residual variance is estimated as `SS/(n-p)`, where `SS` is the minimized sum of squares, `n` is the number of observations, and `p` is the number of estimated parameters. These variance estimates are carried forward to Step 7, where the selected parameters from all variable groups are fitted jointly using a weighted objective.

In [ ]:
import math
import time

from pcse.input import WOFOST81SiteDataProvider_SNOMIN

from diffwofost.physical_models.crop.wofost81 import Wofost81
from diffwofost.physical_models.soil.soil_wrappers import SoilModuleWrapper_NWLP_MLWB_SNOMIN

# Halve the learning rate when the sum of squares stalls. The ceiling is MAX_STEPS.
MAX_STEPS = 80
GROUP_LR = 0.12
PFS = (-1.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.4, 2.7, 3.0, 3.3, 3.7, 4.0, 4.2, 6.0)
CN_RATIO = 10.0
ROOT_TARGET = 125.0
PROTEIN_FACTOR = 5.7
GROWTH_BOUNDS = {
    "AMAX_REF": (20.0, 70.0),
    "SPAN": (15.0, 55.0),
    "TDWI": (50.0, 800.0),
    "RGRLAI": (0.003, 0.02),
    "RNUPTAKEMAX": (1.0, 8.0),
    "NMAXSO": (0.012, 0.030),
    "REALLOC_STEM_FRACTION": (0.05, 0.60),
    "CVO": (0.55, 0.85),
    "NRESIDLV": (0.001, 0.012),
    "NRESIDST": (0.0005, 0.005),
    "TCNT": (2.0, 25.0),
}
BOUNDS.update(GROWTH_BOUNDS)
GROUPS = (
    {"name": "plant_biomass", "major": ("AMAX_REF", "SPAN"), "candidates": ("TDWI", "RGRLAI"), "kind": "biomass"},
    {"name": "plant_N_content", "major": ("RNUPTAKEMAX",), "candidates": ("NMAXSO",), "kind": "nitrogen"},
    {"name": "yield", "major": ("REALLOC_STEM_FRACTION",), "candidates": ("CVO",), "kind": "yield"},
    {"name": "seed_protein", "major": ("NMAXSO",), "candidates": ("NRESIDLV", "NRESIDST", "TCNT"), "kind": "protein"},
)
full_config = Configuration(
    CROP=Wofost81,
    SOIL=SoilModuleWrapper_NWLP_MLWB_SNOMIN,
    OUTPUT_VARS=["DVS", "TAGP", "TWSO", "NamountLV", "NamountST", "NamountSO"],
)


def wosten(clay, silt, organic, density, topsoil):
    """Van Genuchten parameters from the Wösten pedotransfer."""
    clay = max(float(clay), 1.0)
    silt = max(float(silt), 1.0)
    organic = max(float(organic), 0.01)
    density = float(density)
    top = 1.0 if topsoil else 0.0
    t_alpha = (
        -14.96 + 0.03135 * clay + 0.0351 * silt + 0.646 * organic + 15.29 * density
        - 0.192 * top - 4.671 * density**2 - 0.000781 * clay**2 - 0.00687 * organic**2
        + 0.0449 / organic + 0.0663 * math.log(silt) + 0.1482 * math.log(organic)
        - 0.04546 * density * silt - 0.4852 * density * organic + 0.00673 * top * clay
    )
    t_n = (
        -25.23 - 0.02195 * clay + 0.0074 * silt - 0.1940 * organic + 45.5 * density
        - 7.24 * density**2 + 0.0003658 * clay**2 + 0.002885 * organic**2 - 12.81 / density
        - 0.1524 / silt - 0.01958 / organic - 0.2876 * math.log(silt) - 0.0709 * math.log(organic)
        - 44.6 * math.log(density) - 0.02264 * density * clay + 0.0896 * density * organic
        + 0.00718 * top * clay
    )
    t_lambda = (
        0.0202 + 0.0006193 * clay**2 - 0.001136 * organic**2 - 0.2316 * math.log(organic)
        - 0.03544 * density * clay + 0.00283 * density * silt + 0.0488 * density * organic
    )
    t_ksat = (
        7.755 + 0.0352 * silt + 0.93 * top - 0.967 * density**2 - 0.000484 * clay**2
        - 0.000322 * silt**2 + 0.001 / silt - 0.0748 / organic - 0.643 * math.log(silt)
        - 0.01398 * density * clay - 0.1673 * density * organic + 0.02986 * top * clay
        - 0.03305 * top * silt
    )
    theta_s = (
        0.7919 + 0.001691 * clay - 0.29619 * density - 1.491e-6 * silt**2
        + 8.21e-5 * organic**2 + 0.02427 / clay + 0.01113 / silt + 0.01472 * math.log(silt)
        - 7.33e-5 * organic * clay - 0.000619 * density * clay - 0.001183 * density * organic
        - 0.0001664 * top * silt
    )
    return {
        "alpha": math.exp(t_alpha),
        "n": math.exp(t_n) + 1.0,
        "lam": (10 * math.exp(t_lambda) - 10) / (1 + math.exp(t_lambda)),
        "k_sat": math.exp(t_ksat),
        "theta_s": min(max(theta_s, 0.25), 0.65),
        "theta_r": 0.01,
    }


def _moisture(pf, vg):
    psi = 10.0 ** pf
    n = vg["n"]
    m = 1.0 - 1.0 / n
    return vg["theta_r"] + (vg["theta_s"] - vg["theta_r"]) / (1.0 + (vg["alpha"] * psi) ** n) ** m


def _log_k(pf, vg):
    psi = 10.0 ** pf
    n = vg["n"]
    m = 1.0 - 1.0 / n
    ah = vg["alpha"] * psi
    h1 = (1.0 + ah ** n) ** m
    h2 = ah ** (n - 1.0)
    denom = (1.0 + ah ** n) ** (m * (vg["lam"] + 2.0))
    return math.log10(max(vg["k_sat"] * (h1 - h2) ** 2 / denom, 1e-12))


def retention_tables(vg):
    moistures = [_moisture(pf, vg) for pf in PFS]
    for index in range(1, len(moistures)):
        if moistures[index] >= moistures[index - 1]:
            moistures[index] = moistures[index - 1] - 1e-4
    sm, cond = [], []
    for pf, value in zip(PFS, moistures, strict=True):
        sm.extend((float(pf), max(value, 0.005)))
        cond.extend((float(pf), _log_k(pf, vg)))
    return sm, cond


def cell_number(row, name):
    value = row[name]
    if pd.isna(value) or value == "NA":
        return None
    return float(value)


def soil_layers(row):
    layers = []
    for horizon in range(1, int(row.N_Horizons) + 1):
        thickness = cell_number(row, f"Thickness_H{horizon}")
        if thickness is None:
            continue
        clay = cell_number(row, f"Clay_H{horizon}") or 10.0
        silt = (cell_number(row, f"Fine_silt_H{horizon}") or 0.0) + (
            cell_number(row, f"Coarse_silt_H{horizon}") or 0.0
        )
        organic = cell_number(row, f"Organic_matter_H{horizon}")
        organic = 1.0 if organic is None else organic
        density = cell_number(row, f"Apparent_density_H{horizon}") or 1.4
        ph = cell_number(row, f"Water_pH_H{horizon}") or 7.0
        capacity = cell_number(row, f"Field_capacity_H{horizon}")
        wilting = cell_number(row, f"Wilting_point_H{horizon}")
        vg = wosten(clay, silt, organic, density, topsoil=(horizon == 1))
        sm, cond = retention_tables(vg)
        layers.append({
            "Thickness": thickness,
            "SMfromPF": sm,
            "CONDfromPF": cond,
            "CRAIRC": 0.06,
            "CNRatioSOMI": CN_RATIO,
            "RHOD": density,
            "Soil_pH": ph,
            "FSOMI": organic / 100.0,
            "file_fc": None if capacity is None else capacity / 100.0 * density,
            "ptf_fc": _moisture(2.0, vg),
            "file_wp": None if wilting is None else wilting / 100.0 * density,
            "ptf_wp": _moisture(4.2, vg),
        })
    return layers


def with_root_boundary(layers):
    """Split a layer so the rooting depth lands on a boundary at or nearest 125 cm."""
    total = sum(layer["Thickness"] for layer in layers)
    target = min(ROOT_TARGET, total)
    depth = 0.0
    split = []
    for layer in layers:
        thickness = layer["Thickness"]
        crosses = depth < target < depth + thickness - 0.05
        upper = target - depth
        lower = thickness - upper
        if crosses and upper >= 5.0 and lower >= 5.0:
            split.append({**layer, "Thickness": upper})
            split.append({**layer, "Thickness": lower})
        else:
            split.append(layer)
            if crosses:
                target = depth + thickness
        depth += thickness
    boundaries, depth = [], 0.0
    for layer in split:
        depth += layer["Thickness"]
        boundaries.append(depth)
    if not any(abs(boundary - target) < 0.05 for boundary in boundaries):
        target = min(boundaries, key=lambda boundary: abs(boundary - target))
    return split, target


def fertiliser_fractions(product):
    key = str(product).lower().replace("é", "e").replace("è", "e")
    if "ammonitrate" in key:
        return 0.5, 0.5
    if "18_46" in key or "18-46" in key:
        return 1.0, 0.0
    if "solution" in key:
        return 0.75, 0.25
    raise ValueError(f"Unknown fertiliser product: {product}")


def timed_block(title, signal, events):
    if not events:
        return ""
    lines = [
        "      - event_signal: " + signal,
        f"        name: {title}",
        "        comment: recorded in the AgMIP input file",
        "        events_table:",
    ]
    for day, body in events:
        lines.append(f"          - {day.isoformat()}:")
        lines.extend(f"              {key}: {value}" for key, value in body.items())
    return "\n".join(lines)


def season_agro(start, end, dressings, irrigations):
    timed = "\n".join(part for part in (
        timed_block("Fertiliser", "apply_n_snomin", dressings),
        timed_block("Irrigation", "irrigate", irrigations),
    ) if part)
    text = f"""
AgroManagement:
- {start.isoformat()}:
    CropCalendar:
        crop_name: winterwheat
        variety_name: Arminda
        crop_start_date: {start.isoformat()}
        crop_start_type: sowing
        crop_end_date: {end.isoformat()}
        crop_end_type: harvest
        max_duration: 360
    TimedEvents:
{timed or "      []"}
    StateEvents: null
"""
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as handle:
        handle.write(text)
        path = Path(handle.name)
    try:
        return YAMLAgroManagementReader(str(path))
    finally:
        path.unlink(missing_ok=True)


def dressing_events(row, start, end):
    events = []
    count = 0 if pd.isna(row.Number_of_fertilizer_doses) else int(row.Number_of_fertilizer_doses)
    for index in range(1, count + 1):
        day = parse_date(getattr(row, f"Date_fertilization_{index}"))
        amount = cell_number(row, f"Amount_fertilization_{index}")
        product = getattr(row, f"Product_fertilization_{index}")
        if day is None or amount is None or not (start <= day <= end):
            continue
        ammonium, nitrate = fertiliser_fractions(product)
        events.append((day, {
            "amount": amount,
            "application_depth": 10,
            "cnratio": 0,
            "initial_age": 0,
            "f_NH4N": ammonium,
            "f_NO3N": nitrate,
            "f_orgmat": 0,
        }))
    return events


def irrigation_events(row, start, end):
    events = []
    count = 0 if pd.isna(row.Number_of_irrigations) else int(row.Number_of_irrigations)
    for index in range(1, count + 1):
        day = parse_date(getattr(row, f"Date_irrigation_{index}"))
        amount = cell_number(row, f"Amount_irrigation_{index}")
        if day is None or amount is None or not (start <= day <= end):
            continue
        events.append((day, {"amount": amount / 10.0, "efficiency": 1.0}))
    return events


def season_rows(daily, latitude, altitude, start, end):
    window = daily[(daily["DAY"] >= start) & (daily["DAY"] <= end)]
    rows = []
    for record in window.itertuples(index=False):
        tmin = float(record.TMIN)
        tmax = float(record.TMAX)
        temp = 0.5 * (tmin + tmax)
        # The sheet stores ET0 in mm day-1. E0 and ES0 are not in the file.
        reference = float(record.ET0) / 10.0
        rows.append({
            "DAY": record.DAY,
            "LAT": latitude,
            "LON": 2.0,
            "ELEV": altitude,
            "IRRAD": float(record.SRAD) * 1.0e6,
            "TMIN": tmin,
            "TMAX": tmax,
            "TEMP": temp,
            "DTEMP": 0.5 * (temp + tmax),
            "RAIN": float(record.RAIN) / 10.0,
            "VAP": 10.0,
            "WIND": 2.0,
            "ET0": reference,
            "E0": reference,
            "ES0": reference,
        })
    return rows


def harvest_value(block, column, missing_zero=False):
    rows = block.dropna(subset=[column])
    if rows.empty:
        return None
    value = float(rows.iloc[-1][column])
    if missing_zero and value == 0.0:
        return None
    return value


def build_season(number, weather_cache):
    block = observations.loc[observations["Number"] == number]
    site = management.loc[management["Number"] == number].iloc[0]
    sowing = parse_date(block.iloc[0].SowingDate)
    harvest = parse_date(block.dropna(subset=["HarvestDate"]).iloc[-1].HarvestDate)
    station = int(site.Station)
    if station not in weather_cache:
        weather_cache[station] = station_weather(station)
    daily, latitude, altitude = weather_cache[station]
    end = min(harvest, daily["DAY"].max())
    layers, root_depth = with_root_boundary(soil_layers(site))
    clean = []
    for layer in layers:
        clean.append({key: value for key, value in layer.items() if key not in {"file_fc", "ptf_fc", "file_wp", "ptf_wp"}})
    description = {
        "PFWiltingPoint": 4.2,
        "PFFieldCapacity": 2.0,
        "SurfaceConductivity": 70.0,
        "SoilLayers": clean,
        "SubSoilType": {**clean[-1], "Thickness": 200.0, "FSOMI": 0.0},
        "GroundWater": None,
    }
    wav = cell_number(site, "Available_soil_water,_volumetric")
    wav = 20.0 if wav is None else min(wav / 10.0, 100.0)
    n_layers = len(clean)
    site_data = WOFOST81SiteDataProvider_SNOMIN(
        WAV=wav, CO2=400.0, A0SOM=24.0,
        NH4I=[2.0] + [0.0] * (n_layers - 1),
        NO3I=[10.0] + [0.0] * (n_layers - 1),
    )
    season_provider = ParameterProvider(
        cropdata=crop, soildata={"RDMSOL": root_depth, "SoilProfileDescription": description}, sitedata=site_data
    )
    season_provider.set_active_crop("winterwheat", "Arminda", "sowing", "harvest")
    biomass = []
    for record in block.itertuples(index=False):
        if pd.isna(record.Biomass):
            continue
        day = (parse_date(record.Date) - sowing).days
        biomass.append((day, float(record.Biomass) * 10.0))
    return {
        "number": int(number),
        "provider": season_provider,
        "rows": season_rows(daily, latitude, altitude, sowing, end),
        "agro": season_agro(sowing, end, dressing_events(site, sowing, end), irrigation_events(site, sowing, end)),
        "end": end,
        "rdmcr": root_depth,
        "biomass": biomass,
        "yield_gm2": harvest_value(block, "Grain_Yield"),
        "nitrogen": harvest_value(block, "N_in_biomassHarvest"),
        "protein": harvest_value(block, "ProteinContentGrain", missing_zero=True),
        "checks": [(layer["file_fc"], layer["ptf_fc"], layer["file_wp"], layer["ptf_wp"]) for layer in layers],
    }


weather_cache = {}
seasons = [build_season(number, weather_cache) for number in sorted(observations["Number"].unique())]
fc_gap, wp_gap = [], []
for season in seasons:
    for file_fc, ptf_fc, file_wp, ptf_wp in season["checks"]:
        if file_fc is not None:
            fc_gap.append(abs(file_fc - ptf_fc))
        if file_wp is not None:
            wp_gap.append(abs(file_wp - ptf_wp))
print(
    f"Seasons: {len(seasons)}. "
    f"Biomass dates: {sum(len(season['biomass']) for season in seasons)}. "
    f"Yield: {sum(season['yield_gm2'] is not None for season in seasons)}. "
    f"Plant N: {sum(season['nitrogen'] is not None for season in seasons)}. "
    f"Protein: {sum(season['protein'] is not None for season in seasons)}."
)
print(
    f"Pedotransfer versus the file, mean absolute volumetric gap: "
    f"field capacity {sum(fc_gap)/len(fc_gap):.3f}, wilting point {sum(wp_gap)/len(wp_gap):.3f}."
)

current = {name: float(accepted[name]) for name in accepted}
for name in GROWTH_BOUNDS:
    current[name] = float(provider[name])
arminda_values = {name: defaults[name] for name in ("TSUM1", "TSUM2", "VERNSAT", "VERNBASE", "DLO")}
arminda_values.update({name: float(provider[name]) for name in GROWTH_BOUNDS})


def merged(active):
    values = {name: torch.tensor(float(current[name]), dtype=torch.float64) for name in current}
    values.update(active)
    return values


def simulate(season, values):
    season_provider = season["provider"]
    for name, value in values.items():
        if name in BOUNDS:
            season_provider.set_override(name, value, check=False)
    season_provider.set_override("RDMCR", season["rdmcr"], check=False)
    engine = Engine(full_config)
    engine.setup(season_provider, iter(season["rows"]), season["agro"])
    engine.run_till(season["end"])
    output = engine.get_output()
    return {name: torch.stack([day[name].reshape(()) for day in output]) for name in full_config.OUTPUT_VARS}


def season_ss(season, values, kind):
    output = simulate(season, values)
    if kind == "biomass":
        total = output["TAGP"].new_zeros(())
        for day, observed in season["biomass"]:
            if day < 0 or day >= output["TAGP"].shape[0]:
                continue
            total = total + (torch.log(output["TAGP"][day].clamp(min=1.0)) - math.log(observed)).square()
        return total
    if kind == "yield":
        if season["yield_gm2"] is None:
            return output["TWSO"].new_zeros(())
        return (output["TWSO"][-1] / 10.0 - season["yield_gm2"]).square()
    if kind == "nitrogen":
        if season["nitrogen"] is None:
            return output["TAGP"].new_zeros(())
        crop_n = output["NamountLV"][-1] + output["NamountST"][-1] + output["NamountSO"][-1]
        return (100.0 * crop_n / output["TAGP"][-1].clamp(min=1.0) - season["nitrogen"]).square()
    if season["protein"] is None:
        return output["TWSO"].new_zeros(())
    predicted = 100.0 * PROTEIN_FACTOR * output["NamountSO"][-1] / output["TWSO"][-1].clamp(min=1.0)
    return (predicted - season["protein"]).square()


def observation_count(kind):
    if kind == "biomass":
        return sum(len(season["biomass"]) for season in seasons)
    key = {"yield": "yield_gm2", "nitrogen": "nitrogen", "protein": "protein"}[kind]
    return sum(season[key] is not None for season in seasons)


def evaluate_kind(kind, values):
    frozen = {name: torch.tensor(float(values[name]), dtype=torch.float64) for name in values}
    total = 0.0
    with torch.no_grad():
        for season in seasons:
            total += float(season_ss(season, frozen, kind))
    return total


def _values(names, mapping):
    return ", ".join(f"{name} {float(mapping[name]):.4g}" for name in names)


def fit_kind(names, kind, max_steps=MAX_STEPS):
    """Adam on `names`, halving the rate when the sum of squares stalls."""
    pack = ParameterSet(names, current)
    lr = GROUP_LR
    optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
    best_ss = float("inf")
    best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
    waited, cuts, rewound = 0, 0, False
    print(f"  Start: {_values(names, current)}. A dot is one site-year.", flush=True)
    for step in range(max_steps):
        step_started = time.perf_counter()
        optimizer.zero_grad()
        total = 0.0
        print(f"  step {step + 1}/{max_steps} ", end="", flush=True)
        for season in seasons:
            piece = season_ss(season, merged(pack()), kind)
            if piece.requires_grad:
                piece.backward()
                total += float(piece.detach())
            print(".", end="", flush=True)
        elapsed = time.perf_counter() - step_started
        waited, cuts, rewound, action = rate_decision(total, best_ss, waited, cuts, rewound)
        if action == "improve":
            best_ss = total
            best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
            mark = "best"
        elif action == "decay":
            pack.load_state_dict(best_state)
            lr *= LR_FACTOR
            optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
            print(f"  SS {total:.4g}. {elapsed:.0f} s", flush=True)
            print(f"  Learning rate {lr:.4g}. Back at the best step.", flush=True)
            continue
        elif action == "stop":
            print(f"  SS {total:.4g}. {elapsed:.0f} s", flush=True)
            print(f"  Early stopping at step {step}.", flush=True)
            break
        else:
            mark = f"no gain for {waited}"
        print(f"  SS {total:.4g}. {mark}. {elapsed:.0f} s", flush=True)
        optimizer.step()
    else:
        print(f"  Reached the ceiling of {max_steps} steps.", flush=True)
    pack.load_state_dict(best_state)
    print("  Scoring the best step.", flush=True)
    frozen = {name: float(value.detach()) for name, value in pack().items()}
    return frozen, evaluate_kind(kind, {**current, **frozen})
if DO_NOT_RERUN:
    stored = load_results()
    selected = list(stored["selected"])
    current = {name: float(value) for name, value in stored["after_step7"].items()}
    after_step6 = {name: float(value) for name, value in stored["after_step6"].items()}
    group_rows = stored["group_rows"]
    group_variance = {key: float(value) for key, value in stored["group_variance"].items()}
    print("DO_NOT_RERUN is set, so the group fits and the weighted refit were loaded.")
else:
    phenology_selected = list(selected)
    group_rows = []
    group_variance = {"phenology": float(error_variance)}
    for spec in GROUPS:
        kind = spec["kind"]
        n_obs = observation_count(kind)
        free_majors = [name for name in spec["major"] if name not in selected]
        free_candidates = [name for name in spec["candidates"] if name not in selected]
        print(
            f"\n{spec['name']} ({kind}). n = {n_obs}. "
            f"Majors: {', '.join(free_majors) or 'already selected'}. "
            f"Candidates: {', '.join(free_candidates) or 'none'}.",
            flush=True,
        )
        chosen = []
        if free_majors:
            fitted, ss = fit_kind(free_majors, kind)
            current.update(fitted)
            chosen = list(free_majors)
        else:
            print("  Scoring the group at the parameters already selected.", flush=True)
            ss = evaluate_kind(kind, current)
        best_ss = ss
        best_score = None
        if chosen:
            best_score = float(aicc(torch.tensor(ss), n_obs, len(chosen)))
            group_rows.append({
                "Group": spec["name"], "Parameters": ", ".join(chosen),
                "SS": ss, "AICc": best_score, "Kept": "major",
            })
            print(f"  Majors kept. SS {ss:.1f} at the updated parameters, p = {len(chosen)}, AICc {best_score:.2f}.")
        for candidate in free_candidates:
            trial = chosen + [candidate]
            fitted, ss = fit_kind(trial, kind)
            score = float(aicc(torch.tensor(ss), n_obs, len(trial)))
            keep = best_score is None or score < best_score
            group_rows.append({
                "Group": spec["name"], "Parameters": ", ".join(trial),
                "SS": ss, "AICc": score, "Kept": "yes" if keep else "no",
            })
            if best_score is None:
                comparison = f"AICc {score:.2f}"
            else:
                comparison = f"AICc {score:.2f}, change {score - best_score:+.2f}"
            decision = "Kept." if keep else "Left at the default."
            print(f"  {candidate}: SS {ss:.1f} at the trial parameters, {comparison}. {decision}")
            if keep:
                current.update(fitted)
                chosen = trial
                best_ss = ss
                best_score = score
        if chosen:
            selected.extend(name for name in chosen if name not in selected)
        denom = n_obs - len(chosen)
        group_variance[kind] = best_ss / denom if denom > 0 else best_ss / n_obs
        held = _values(chosen, current) if chosen else "unchanged"
        print(f"  Weight SS/(n-p) = {group_variance[kind]:.4g}. {held}.")

    print("\nWeighted refit of every selected parameter.", flush=True)


    def fit_weighted(max_steps=MAX_STEPS):
        pack = ParameterSet(selected, current)
        lr = GROUP_LR
        optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
        best_ss = float("inf")
        best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
        waited, cuts, rewound = 0, 0, False
        print(f"  {', '.join(selected)}.", flush=True)
        print("  Phenology first, then one dot per site-year in each group.", flush=True)
        for step in range(max_steps):
            step_started = time.perf_counter()
            optimizer.zero_grad()
            total = 0.0
            print(f"  weighted step {step + 1}/{max_steps}", flush=True)
            print("    phenology", end="", flush=True)
            pheno = sum_of_squares(pack()) / group_variance["phenology"]
            pheno.backward()
            total += float(pheno.detach())
            print(f" weighted SS {float(pheno.detach()):.2f}", flush=True)
            for spec in GROUPS:
                variance = group_variance[spec["kind"]]
                print(f"    {spec['name']} ", end="", flush=True)
                for season in seasons:
                    piece = season_ss(season, merged(pack()), spec["kind"]) / variance
                    if piece.requires_grad:
                        piece.backward()
                        total += float(piece.detach())
                    print(".", end="", flush=True)
                print(flush=True)
            elapsed = time.perf_counter() - step_started
            waited, cuts, rewound, action = rate_decision(total, best_ss, waited, cuts, rewound)
            if action == "improve":
                best_ss = total
                best_state = {key: value.detach().clone() for key, value in pack.state_dict().items()}
                mark = "best"
            elif action == "decay":
                pack.load_state_dict(best_state)
                lr *= LR_FACTOR
                optimizer = torch.optim.Adam(pack.parameters(), lr=lr)
                print(f"    objective {total:.2f}. {elapsed / 60:.1f} min.", flush=True)
                print(f"  Learning rate {lr:.4g}. Back at the best step.", flush=True)
                continue
            elif action == "stop":
                print(f"    objective {total:.2f}. {elapsed / 60:.1f} min.", flush=True)
                print(f"  Early stopping at step {step}.", flush=True)
                break
            else:
                mark = f"no gain for {waited}"
            print(
                f"    objective {total:.2f}. {mark}. {elapsed / 60:.1f} min.",
                flush=True,
            )
            optimizer.step()
        else:
            print(f"  Reached the ceiling of {max_steps} steps.", flush=True)
        pack.load_state_dict(best_state)
        return {name: float(value.detach()) for name, value in pack().items()}
    after_step6 = dict(current)
    current.update(fit_weighted())
    print("\nGroup decisions", flush=True)
    for row in group_rows:
        print(f"  {row['Group']}: {row['Parameters']}  SS {row['SS']:.1f}  AICc {row['AICc']:.2f}  {row['Kept']}")
    left = [name for name in GROWTH_BOUNDS if name not in selected]
    print("Selected:", ", ".join(selected))
    print("Growth parameters left at Arminda:", ", ".join(left) or "none")
    save_results({
        "phenology_history": history,
        "phenology_accepted": accepted,
        "phenology_selected": phenology_selected,
        "phenology_error_variance": error_variance,
        "selected": selected,
        "after_step6": after_step6,
        "after_step7": current,
        "group_rows": group_rows,
        "group_variance": group_variance,
    })
pd.DataFrame(group_rows)


## Step 8 on the calibration site-years

These 14 site-years are the observations the parameters were fitted to. The table gives bias, RMSE and RRMSE at the Arminda default, after the separate group fits, and after the weighted refit. Bias is simulated minus observed. RRMSE is the RMSE divided by the mean of the observations. Biomass is reported in g m-2 and as its logarithm, the scale used in that group's sum of squares. The figure shows the same site-years for the Arminda default and the final parameters. The eight evaluation environments stay out of this comparison: their observations are not in the public file.
The parameter vectors are the stored Snellius search. Running the next cell draws this table and figure from those vectors. The previous short search is no longer the one shown here.


In [ ]:
import matplotlib.pyplot as plt

STAGE = {"BBCH55": 1.0, "BBCH90": 2.0}
OBSERVED_STAGE = {
    environment["number"]: {"BBCH55": environment["heading"], "BBCH90": environment["maturity"]}
    for environment in environments
}
VARIABLES = ["BBCH55", "BBCH90", "biomass", "ln(biomass)", "plant N", "yield", "protein"]
PLOT_VARIABLES = ["BBCH55", "BBCH90", "biomass", "plant N", "yield", "protein"]
UNITS = {
    "BBCH55": "days after sowing",
    "BBCH90": "days after sowing",
    "biomass": "g m-2",
    "plant N": "% N",
    "yield": "g m-2",
    "protein": "% protein",
}


def calibration_pairs(values):
    """Observed and simulated values on the calibration site-years."""
    frozen = {name: torch.tensor(float(values[name]), dtype=torch.float64) for name in values}
    rows = []
    missed = []
    with torch.no_grad():
        for season in seasons:
            output = simulate(season, frozen)
            number = season["number"]
            for name, threshold in STAGE.items():
                day = hard_day(output["DVS"], threshold)
                observed = OBSERVED_STAGE[number][name]
                if day is None:
                    missed.append((number, name))
                    continue
                rows.append({"variable": name, "observed": observed, "simulated": day, "number": number})
            for day, observed_kg in season["biomass"]:
                if day < 0 or day >= output["TAGP"].shape[0]:
                    continue
                observed = observed_kg / 10.0
                simulated = float(output["TAGP"][day]) / 10.0
                rows.append({"variable": "biomass", "observed": observed, "simulated": simulated, "number": number})
                rows.append({
                    "variable": "ln(biomass)",
                    "observed": math.log(observed),
                    "simulated": math.log(max(simulated, 0.1)),
                    "number": number,
                })
            if season["yield_gm2"] is not None:
                rows.append({
                    "variable": "yield",
                    "observed": season["yield_gm2"],
                    "simulated": float(output["TWSO"][-1]) / 10.0,
                    "number": number,
                })
            if season["nitrogen"] is not None:
                crop_n = float(output["NamountLV"][-1] + output["NamountST"][-1] + output["NamountSO"][-1])
                predicted = 100.0 * crop_n / max(float(output["TAGP"][-1]), 1.0)
                rows.append({"variable": "plant N", "observed": season["nitrogen"], "simulated": predicted, "number": number})
            if season["protein"] is not None:
                predicted = 100.0 * PROTEIN_FACTOR * float(output["NamountSO"][-1]) / max(float(output["TWSO"][-1]), 1.0)
                rows.append({"variable": "protein", "observed": season["protein"], "simulated": predicted, "number": number})
    if missed:
        print("Stage not reached: " + ", ".join(f"{number} {name}" for number, name in missed))
    return pd.DataFrame(rows)


def agreement_rows(pairs, label):
    rows = []
    for variable in VARIABLES:
        block = pairs.loc[pairs["variable"] == variable]
        if block.empty:
            rows.append({"Variable": variable, "Parameters": label, "n": 0, "bias": None, "RMSE": None, "RRMSE": None})
            continue
        error = block["simulated"] - block["observed"]
        rmse = float((error.pow(2).mean()) ** 0.5)
        scale = float(block["observed"].abs().mean())
        rows.append({
            "Variable": variable,
            "Parameters": label,
            "n": int(len(block)),
            "bias": float(error.mean()),
            "RMSE": rmse,
            "RRMSE": rmse / scale,
        })
    return rows


parameter_sets = (
    ("Arminda default", arminda_values),
    ("after step 6", after_step6),
    ("after step 7", current),
)
calibration_points = {}
rows = []
for label, values in parameter_sets:
    print(f"Calibration agreement, {label}.")
    calibration_points[label] = calibration_pairs(values)
    rows.extend(agreement_rows(calibration_points[label], label))
agreement = pd.DataFrame(rows)
display(agreement.round(3))

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, variable in zip(axes.ravel(), PLOT_VARIABLES):
    limits = []
    for label, marker in (("Arminda default", "o"), ("after step 7", "s")):
        block = calibration_points[label]
        block = block.loc[block["variable"] == variable]
        if block.empty:
            continue
        ax.scatter(block["observed"], block["simulated"], marker=marker, s=28, label=label)
        limits.extend(block["observed"].tolist())
        limits.extend(block["simulated"].tolist())
    if not limits:
        ax.set_title(variable)
        continue
    low, high = min(limits), max(limits)
    pad = 0.05 * (high - low or 1.0)
    ax.plot([low - pad, high + pad], [low - pad, high + pad], color="0.5", linewidth=1)
    ax.set_title(variable)
    ax.set_xlabel(f"observed, {UNITS[variable]}")
    ax.set_ylabel("simulated")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Calibration site-years")
fig.tight_layout()
plt.show()


## Simulated versus observed, as in the supplement

Supplementary Figs. S1–S3 are simulated against observed on the calibration data: one figure for the default parameters, one after the group-wise fits, and one after the weighted refit. The three figures below are that comparison for WOFOST. The STICS figures also have panels for days to stem elongation and grain number. WOFOST has no development-stage switch for stem elongation and no grain-number state, so those two panels are absent. Biomass is the logarithm, which is the biomass variable in their goodness-of-fit table.

In [ ]:
PAPER_VARIABLES = ["BBCH55", "BBCH90", "ln(biomass)", "plant N", "yield", "protein"]
PAPER_UNITS = {
    "BBCH55": "days after sowing",
    "BBCH90": "days after sowing",
    "ln(biomass)": "ln(g m-2)",
    "plant N": "% N",
    "yield": "g m-2",
    "protein": "% protein",
}


def draw_simulated_observed(label):
    """One simulated-versus-observed figure for a single parameter vector."""
    points = calibration_points[label]
    fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
    for ax, variable in zip(axes.ravel(), PAPER_VARIABLES):
        block = points.loc[points["variable"] == variable]
        if block.empty:
            ax.set_title(variable)
            continue
        ax.scatter(block["observed"], block["simulated"], s=28, color="0.15")
        limits = block["observed"].tolist() + block["simulated"].tolist()
        low, high = min(limits), max(limits)
        pad = 0.05 * (high - low or 1.0)
        ax.plot([low - pad, high + pad], [low - pad, high + pad], color="0.55", linewidth=1)
        ax.set_title(variable)
        ax.set_xlabel(f"observed, {PAPER_UNITS[variable]}")
        ax.set_ylabel("simulated")
    fig.suptitle(label)
    fig.tight_layout()
    plt.show()


for label, _values in parameter_sets:
    draw_simulated_observed(label)


## In-season biomass, crop nitrogen and soil mineral nitrogen

Biomass was weighed several times between March and harvest on 10 of the 14 calibration site-years. The circles on the first figure are those measurements. Plant nitrogen was measured only at harvest. Grey vertical lines are the mineral-fertilizer dressings. This file has no manure, and these calibration sites were not irrigated.

Aboveground biomass is living plus dead leaves, stems and grain. When a leaf class reaches its lifespan, its weight moves from the living pool to the dead pool, so the biomass line does not step down. The nitrogen in that class is taken off the living-leaf pool at the concentration of that day and counted as a loss. It is not stored in the dead leaves, and it is not returned to the soil during the season. Crop nitrogen percentage therefore steps down, while the soil curve does not step up. Nitrogen moved into the grain stays in the aboveground total.

The soil curve is ammonium plus nitrate in the whole profile. A dressing raises it by about the applied rate on the day after the grey line.

In [ ]:
import matplotlib.pyplot as plt

for name in ("NH4T", "NO3T"):
    if name not in full_config.OUTPUT_VARS:
        full_config.OUTPUT_VARS.append(name)


def fertiliser_days(number, sowing):
    """Dressing dates as days after sowing, and the nitrogen rate in kg ha-1."""
    row = management.loc[management["Number"] == number].iloc[0]
    count = 0 if pd.isna(row.Number_of_fertilizer_doses) else int(row.Number_of_fertilizer_doses)
    days = []
    for index in range(1, count + 1):
        day = parse_date(getattr(row, f"Date_fertilization_{index}"))
        amount = cell_number(row, f"Amount_fertilization_{index}")
        if day is None or amount is None:
            continue
        days.append(((day - sowing).days, amount))
    return days


def observed_harvest_nitrogen(number, sowing):
    block = observations.loc[observations["Number"] == number].dropna(subset=["N_in_biomassHarvest"])
    if block.empty:
        return None
    return (parse_date(block.iloc[0]["Date"]) - sowing).days, float(block.iloc[0]["N_in_biomassHarvest"])


def trajectory(values):
    """Daily biomass, crop nitrogen percent and profile mineral nitrogen."""
    frozen = {name: torch.tensor(float(values[name]), dtype=torch.float64) for name in values}
    curves = {}
    with torch.no_grad():
        for season in seasons:
            output = simulate(season, frozen)
            sowing = season["rows"][0]["DAY"]
            tagp = output["TAGP"].detach()
            crop_n = output["NamountLV"] + output["NamountST"] + output["NamountSO"]
            curves[season["number"]] = {
                "biomass": tagp / 10.0,
                "nitrogen": 100.0 * crop_n.detach() / tagp.clamp(min=1.0),
                "soil_n": (output["NH4T"] + output["NO3T"]).detach(),
                "fertiliser": fertiliser_days(season["number"], sowing),
                "observed_biomass": [(day, kg / 10.0) for day, kg in season["biomass"]],
                "observed_nitrogen": observed_harvest_nitrogen(season["number"], sowing),
                "observed_soil": None,
            }
    return curves


print("In-season curves, Arminda default.")
arminda_curves = trajectory(arminda_values)
print("In-season curves, after step 7.")
final_curves = trajectory(current)
numbers = sorted(final_curves)


def draw_trajectories(key, observed_key, ylabel, title):
    fig, axes = plt.subplots(3, 5, figsize=(12, 7), sharex=False)
    for ax, number in zip(axes.ravel(), numbers):
        final = final_curves[number]
        arminda = arminda_curves[number]
        days = torch.arange(final["biomass"].shape[0])
        for day, _amount in final["fertiliser"]:
            ax.axvline(day, color="0.75", linewidth=0.8, zorder=0)
        ax.plot(days, arminda[key], color="0.45", linewidth=1, label="Arminda default")
        ax.plot(days, final[key], color="C0", linewidth=1.4, label="after step 7")
        observed = final[observed_key]
        if observed_key == "observed_biomass" and observed:
            ax.scatter(
                [day for day, _value in observed],
                [value for _day, value in observed],
                s=18, color="black", zorder=3, label="observed",
            )
        elif observed_key != "observed_biomass" and observed is not None:
            ax.scatter([observed[0]], [observed[1]], s=22, color="black", zorder=3, label="observed")
        ax.set_title(str(number), fontsize=9)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(numbers):]:
        ax.axis("off")
    axes[0, 0].legend(frameon=False, fontsize=7)
    fig.suptitle(title)
    fig.supxlabel("days after sowing")
    fig.supylabel(ylabel)
    fig.tight_layout()
    plt.show()


draw_trajectories(
    "biomass", "observed_biomass", "g m-2",
    "Aboveground biomass. Grey lines are fertilizer dressings.",
)
draw_trajectories(
    "nitrogen", "observed_nitrogen", "% N",
    "Aboveground nitrogen. The circle is the harvest measurement.",
)
draw_trajectories(
    "soil_n", "observed_soil", "kg N ha-1",
    "Mineral nitrogen in the profile. Grey lines are fertilizer dressings.",
)


## Adam and Nelder–Mead

The tables are the Snellius searches copied on 26 October 2026. Both methods kept the same parameters. The values differ, and so do the sums of squares. A candidate is kept when its AICc is lower than the round before it.

Phenology, n = 28. Twenty starts for `TSUM1` and `TSUM2`, then five for each candidate. Every Nelder–Mead start met its stopping test.

| Parameters | Adam SS | Adam AICc | Adam | Nelder–Mead SS | Nelder–Mead AICc | Nelder–Mead |
|---|---:|---:|---|---:|---:|---|
| TSUM1, TSUM2 | 839.05 | 99.68 | kept | 837.88 | 99.64 | kept |
| TSUM1, TSUM2, VERNSAT | 839.05 | 102.20 | left out | 837.88 | 102.16 | left out |
| TSUM1, TSUM2, VERNBASE | 839.05 | 102.20 | left out | 837.88 | 102.16 | left out |
| TSUM1, TSUM2, DLO | 839.05 | 102.20 | left out | 837.88 | 102.16 | left out |

Adam's phenology values are `TSUM1` 618.4 °C day and `TSUM2` 956.1 °C day. Nelder–Mead's are 618.4 and 961.4.

Biomass is the log of `TAGP`, nitrogen is the harvest nitrogen percentage, yield is g m-2, and protein is the percentage in the grain. Each row is that group's own sum of squares.

| Group | Parameters | Adam SS | Adam AICc | Adam | Nelder–Mead SS | Nelder–Mead AICc | Nelder–Mead |
|---|---|---:|---:|---|---:|---:|---|
| plant_biomass | AMAX_REF, SPAN | 1.638 | −140.49 | kept | 1.637 | −140.52 | kept |
| plant_biomass | AMAX_REF, SPAN, TDWI | 1.343 | −146.92 | kept | 1.274 | −149.24 | kept |
| plant_biomass | AMAX_REF, SPAN, TDWI, RGRLAI | 1.340 | −144.59 | left out | 1.274 | −146.81 | left out |
| plant_N_content | RNUPTAKEMAX | 0.398 | −25.49 | kept | 0.279 | −28.69 | kept |
| plant_N_content | RNUPTAKEMAX, NMAXSO | 0.379 | −22.51 | left out | 0.261 | −25.86 | left out |
| yield | REALLOC_STEM_FRACTION | 355936 | 135.19 | kept | 203051 | 127.90 | kept |
| yield | REALLOC_STEM_FRACTION, CVO | 340437 | 137.45 | left out | 191218 | 129.95 | left out |
| seed_protein | NMAXSO | 10.259 | −0.71 | kept | 10.419 | −0.51 | kept |
| seed_protein | NMAXSO, NRESIDLV | 9.842 | 1.58 | left out | 9.728 | 1.43 | left out |
| seed_protein | NMAXSO, NRESIDST | | | cut off at step 37 | 10.115 | 1.94 | left out |
| seed_protein | NMAXSO, TCNT | | | not started | 9.961 | 1.74 | left out |

The kept values, with Arminda beside them. Adam's step 7 column is the best weighted start that had finished: start 1, stopped by patience, joint score 98.06 on Adam's own weights. It sits on the step 6 vector. Nelder–Mead's weighted refit had not finished a start, so it has no step 7 column. The two joint scores use different weights and are not a comparison with each other.

| Parameter | Arminda | Adam step 6 | Adam step 7 | Nelder–Mead step 6 |
|---|---:|---:|---:|---:|
| TSUM1 | 886 | 618.43 | 618.45 | 618.45 |
| TSUM2 | 860 | 956.09 | 961.57 | 961.40 |
| AMAX_REF | 40.2 | 29.91 | 29.98 | 31.98 |
| SPAN | 35 | 51.78 | 51.87 | 34.10 |
| TDWI | 593 | 325.1 | 321.0 | 283.7 |
| RNUPTAKEMAX | 4.26 | 1.294 | 1.287 | 1.309 |
| REALLOC_STEM_FRACTION | 0.200 | 0.131 | 0.129 | 0.327 |
| NMAXSO | 0.02035 | 0.01954 | 0.01965 | 0.01946 |

`VERNSAT`, `VERNBASE`, `DLO`, `RGRLAI`, `CVO`, `NRESIDLV`, `NRESIDST` and `TCNT` stay at Arminda in both selections. The next cell reloads these tables from the two JSON files and plots where the 20 phenology starts finished.


In [ ]:
from IPython.display import Markdown
import matplotlib.pyplot as plt
import numpy as np

adam = load_results()
nelder = load_nelder()
phenology = nelder["phenology"]
majors = phenology["rounds"][0]

rows = []
for adam_round, nelder_round in zip(phenology["adam_history"], phenology["rounds"]):
    rows.append(
        {
            "Adam parameters": ", ".join(adam_round["parameters"]),
            "Adam SS": adam_round["ss"],
            "Adam AICc": adam_round["aicc"],
            "Adam kept": adam_round["kept"],
            "Nelder-Mead parameters": ", ".join(nelder_round["names"]),
            "Nelder-Mead SS": nelder_round["best_ss"],
            "Nelder-Mead AICc": nelder_round["aicc"],
            "Nelder-Mead kept": nelder_round["kept"],
            "Starts meeting the stopping test": (
                f"{nelder_round['n_met_stopping_test']} / {nelder_round['n_starts']}"
            ),
        }
    )
display(pd.DataFrame(rows).round(2))

display(Markdown("### Candidate selection"))


def selection_view(frame, ss_col, aicc_col, kept_col, prefix):
    """One method's candidate decisions. ΔAICc is against the best model kept so far."""
    groups = frame[["Group", "Parameters", ss_col, aicc_col, kept_col]].copy()
    groups.columns = ["Group", "Parameters", "SS", "AICc", "Kept"]
    group_names = {
        "plant_biomass": "Biomass",
        "plant_N_content": "Plant nitrogen",
        "yield": "Yield",
        "seed_protein": "Grain protein",
        "phenology": "Phenology",
    }
    groups["Group"] = groups["Group"].map(group_names).fillna(groups["Group"])
    groups["ΔAICc"] = np.nan
    groups["Decision"] = ""
    unfinished = {"cut off at step 37": "Cut off", "not started": "Not started"}
    for _, indices in groups.groupby("Group", sort=False).groups.items():
        best_aicc = None
        for i in indices:
            kept = str(groups.loc[i, "Kept"]).lower()
            aicc_value = groups.loc[i, "AICc"]
            if pd.isna(aicc_value):
                groups.loc[i, "Decision"] = unfinished.get(kept, str(groups.loc[i, "Kept"]))
                continue
            aicc_value = float(aicc_value)
            if kept == "major":
                groups.loc[i, "Decision"] = "Major"
                best_aicc = aicc_value
            else:
                groups.loc[i, "ΔAICc"] = aicc_value - best_aicc
                if kept == "yes":
                    groups.loc[i, "Decision"] = "Retained"
                    best_aicc = aicc_value
                elif kept == "no":
                    groups.loc[i, "Decision"] = "Rejected"
                else:
                    groups.loc[i, "Decision"] = unfinished.get(kept, str(groups.loc[i, "Kept"]))
    groups = groups.rename(columns={"Parameters": "Parameters tested"})
    groups["SS"] = groups["SS"].map(lambda x: "—" if pd.isna(x) else f"{x:.3g}")
    groups["AICc"] = groups["AICc"].map(lambda x: "—" if pd.isna(x) else f"{x:.2f}")
    groups["ΔAICc"] = groups["ΔAICc"].map(lambda x: "—" if pd.isna(x) else f"{x:+.2f}")
    groups = groups.rename(columns={
        "SS": f"{prefix} SS",
        "AICc": f"{prefix} AICc",
        "ΔAICc": f"{prefix} ΔAICc",
        "Decision": prefix,
    })
    return groups[[
        "Group", "Parameters tested",
        f"{prefix} SS", f"{prefix} AICc", f"{prefix} ΔAICc", prefix,
    ]]


stored_groups = pd.DataFrame(nelder["groups"])
candidate_selection = selection_view(
    stored_groups, "Adam SS", "Adam AICc", "Adam kept", "Adam"
).merge(
    selection_view(
        stored_groups, "Nelder-Mead SS", "Nelder-Mead AICc", "Nelder-Mead kept", "Nelder-Mead"
    ),
    on=["Group", "Parameters tested"],
)
display(candidate_selection.reset_index(drop=True))

display(Markdown("### Final parameter estimates"))

names = [
    "TSUM1", "TSUM2", "AMAX_REF", "SPAN", "TDWI",
    "RNUPTAKEMAX", "REALLOC_STEM_FRACTION", "NMAXSO",
]
display(pd.DataFrame(
    {
        "Parameter": names,
        "Arminda": [nelder["arminda"][name] for name in names],
        "Adam step 6": [adam["after_step6"][name] for name in names],
        "Adam step 7": [adam["after_step7"][name] for name in names],
        "Nelder-Mead step 6": [nelder["after_step6"][name] for name in names],
    }
).round(3))
print(
    f"Adam weighted refit, best finished start: SS {adam['weighted_ss']:.2f}. "
    f"Complete: {adam['weighted_complete']}."
)

runs = majors["runs"]
order = np.argsort([run["ss"] for run in runs])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for position, run_index in enumerate(order):
    run = runs[int(run_index)]
    colour = "#1b4f72" if run["met_stopping_test"] else "#c0392b"
    axes[0].scatter(position + 1, run["ss"], color=colour, zorder=3)
axes[0].axhline(majors["best_ss"], color="0.35", lw=0.8)
axes[0].set_xlabel("Start, ordered by final sum of squares")
axes[0].set_ylabel("Final sum of squares")
axes[0].set_title("Phenology, TSUM1 and TSUM2")
met = plt.Line2D([0], [0], marker="o", color="#1b4f72", ls="", label="Met the stopping test")
ceiling = plt.Line2D([0], [0], marker="o", color="#c0392b", ls="", label="Stopped at the evaluation limit")
axes[0].legend(handles=[met, ceiling], fontsize=8, frameon=False)

for run in runs:
    colour = "#1b4f72" if run["met_stopping_test"] else "#c0392b"
    axes[1].scatter(run["final"]["TSUM1"], run["final"]["TSUM2"], color=colour, zorder=3)
axes[1].scatter(
    [majors["best_final"]["TSUM1"]],
    [majors["best_final"]["TSUM2"]],
    facecolors="none",
    edgecolors="black",
    s=80,
    zorder=4,
)
axes[1].set_xlabel("TSUM1 (°C day)")
axes[1].set_ylabel("TSUM2 (°C day)")
axes[1].set_title("Where the 20 starts finished")
fig.tight_layout()
plt.show()


## What this does and does not answer

The agreement figures are the stored Adam vectors, on the calibration site-years, once the notebook is run. Both searches kept `TSUM1`, `TSUM2`, `AMAX_REF`, `SPAN`, `TDWI`, `RNUPTAKEMAX`, `REALLOC_STEM_FRACTION` and `NMAXSO`. Nelder–Mead's biomass and yield sums of squares are the lower ones. Adam's protein candidate `NRESIDST` was cut off at step 37 when the allocation ended, and `TCNT` was never started. Nelder–Mead tried both and left them out. Step 8 stays unscored: the eight evaluation observations are not in the public file.
